# 2.3 Data Exploration — Titanic Survival Prediction

> **Author:** Thibauld
> **Date:** 2026-03-30
> **CRISP-DM Phase:** 2. Data Understanding
> **Purpose:** Perform deeper EDA to uncover patterns, relationships, and anomalies that inform feature engineering and modeling decisions.
>
> **Data Mining Goals (from 1.3):**
> - DM1: Rank features by predictive importance
> - DM2: Train binary classifier achieving >80% accuracy
> - DM3: Compare multiple model families
> - DM4: Measure incremental accuracy gain from engineered features

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Resolve project root — works whether CWD is notebooks/ or project root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "titanic"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Figures dir:  {FIG_DIR}")

# Load data (per 2.1 loading instructions)
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
baseline = pd.read_csv(DATA_DIR / "gender_submission.csv")

print(f"\nTrain: {train.shape[0]} rows, {train.shape[1]} cols")
print(f"Test:  {test.shape[0]} rows, {test.shape[1]} cols")
print(f"Baseline: {baseline.shape[0]} rows, {baseline.shape[1]} cols")
train.head()

## 2. Target Variable Analysis

The target is `Survived` (0 = deceased, 1 = survived). Understanding its distribution is essential for choosing evaluation metrics and CV strategy.

In [ ]:
surv_counts = train['Survived'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
surv_counts.plot(kind='bar', ax=axes[0], color=['#d9534f', '#5cb85c'])
axes[0].set_title('Survived Distribution')
axes[0].set_xlabel('Survived')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Deceased (0)', 'Survived (1)'], rotation=0)
for i, v in enumerate(surv_counts):
    axes[0].text(i, v + 10, f'{v}\n({v/len(train)*100:.1f}%)', ha='center')

# Pie chart
axes[1].pie(surv_counts, labels=['Deceased', 'Survived'], autopct='%1.1f%%',
            colors=['#d9534f', '#5cb85c'], startangle=90)
axes[1].set_title('Survival Rate')

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/target_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Deceased: {surv_counts[0]} ({surv_counts[0]/len(train)*100:.1f}%)")
print(f"Survived: {surv_counts[1]} ({surv_counts[1]/len(train)*100:.1f}%)")
print(f"Ratio (deceased:survived): {surv_counts[0]/surv_counts[1]:.2f}:1")
print(f"\nModerate imbalance — stratified CV required, but resampling not needed.")

## 3. Numeric Feature Distributions

Examine distributions, skewness, and outliers for Age, SibSp, Parch, and Fare. Red flags from 2.2: Age ~20% missing, Fare heavily right-skewed, SibSp/Parch zero-inflated.

In [ ]:
numeric_cols = ['Age', 'SibSp', 'Parch', 'Fare']

# Summary statistics
rows = []
for col in numeric_cols:
    s = train[col].dropna()
    sk = stats.skew(s)
    ku = stats.kurtosis(s)
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    outliers = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    rows.append({
        'Feature': col, 'Mean': f'{s.mean():.2f}', 'Median': f'{s.median():.2f}',
        'Std': f'{s.std():.2f}', 'Skewness': f'{sk:.2f}', 'Kurtosis': f'{ku:.2f}',
        'Outliers (IQR)': outliers, 'Null %': f'{train[col].isna().mean()*100:.1f}%'
    })
pd.DataFrame(rows).set_index('Feature')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(numeric_cols):
    train[col].hist(bins=30, ax=axes[0, i], color='steelblue', edgecolor='white')
    axes[0, i].set_title(f'{col} Distribution')
    axes[0, i].set_xlabel(col)
    train.boxplot(column=col, ax=axes[1, i])
    axes[1, i].set_title(f'{col} Box Plot')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/numeric_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

print("Key observations:")
print("- Age: mild positive skew (0.39), roughly normal — 11 outliers (elderly passengers)")
print("- SibSp: heavily zero-inflated (median=0), extreme kurtosis (17.8)")
print("- Parch: heavily zero-inflated (median=0), most traveled without parents/children")
print("- Fare: extremely right-skewed (4.78) — log transform strongly recommended")

## 4. Categorical Feature Distributions

Examine Pclass, Sex, Embarked, and extracted Title. Title extraction is a key feature engineering hypothesis from 1.3.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['Pclass', 'Sex', 'Embarked']):
    vc = train[col].value_counts()
    vc.plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=0)
    for j, v in enumerate(vc):
        axes[i].text(j, v + 10, f'{v}\n({v/len(train)*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/categorical_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Title extraction from Name field
train['Title'] = train['Name'].str.extract(r',\s*([^\.]+)\.')
title_counts = train['Title'].value_counts()

fig, ax = plt.subplots(figsize=(12, 4))
title_counts.head(10).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Title Distribution (top 10)')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("Title value counts:")
print(title_counts.to_string())
print(f"\nTop 4 titles (Mr/Miss/Mrs/Master) cover {title_counts.head(4).sum()}/{len(train)} = {title_counts.head(4).sum()/len(train)*100:.1f}% of passengers")

## 5. Feature-Target Relationships

Core analysis for DM1 (feature importance). Examine survival rates by Sex, Pclass, Embarked, Title, Age group, and Fare. Include statistical tests to quantify association strength.

In [ ]:
# Survival rates by key categorical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Sex
train.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0,0], color=['#d9534f','#5cb85c'])
axes[0,0].set_title('Survival Rate by Sex')
axes[0,0].set_ylabel('Survival Rate')
axes[0,0].tick_params(axis='x', rotation=0)
axes[0,0].set_ylim(0, 1)

# Pclass
train.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Survival Rate by Pclass')
axes[0,1].set_ylabel('Survival Rate')
axes[0,1].set_ylim(0, 1)

# Embarked
train.groupby('Embarked')['Survived'].mean().plot(kind='bar', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Survival Rate by Embarked')
axes[0,2].set_ylabel('Survival Rate')
axes[0,2].tick_params(axis='x', rotation=0)
axes[0,2].set_ylim(0, 1)

# Title (top 6)
top_titles = ['Mr', 'Mrs', 'Miss', 'Master', 'Dr', 'Rev']
title_surv = train[train['Title'].isin(top_titles)].groupby('Title')['Survived'].mean().reindex(top_titles)
title_surv.plot(kind='bar', ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Survival Rate by Title')
axes[1,0].set_ylabel('Survival Rate')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].set_ylim(0, 1)

# Age by survival (overlaid histograms)
train[train['Survived']==0]['Age'].hist(bins=30, alpha=0.5, ax=axes[1,1], label='Deceased', color='#d9534f')
train[train['Survived']==1]['Age'].hist(bins=30, alpha=0.5, ax=axes[1,1], label='Survived', color='#5cb85c')
axes[1,1].set_title('Age Distribution by Survival')
axes[1,1].legend()

# Fare by survival (log scale)
for surv, color, label in [(0, '#d9534f', 'Deceased'), (1, '#5cb85c', 'Survived')]:
    fares = train[train['Survived']==surv]['Fare']
    fares[fares > 0].apply(np.log1p).hist(bins=30, alpha=0.5, ax=axes[1,2], label=label, color=color)
axes[1,2].set_title('log(Fare+1) by Survival')
axes[1,2].legend()

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/survival_by_features.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed survival rate tables
print("=" * 60)
print("SURVIVAL BY SEX")
print("=" * 60)
for sex, grp in train.groupby('Sex')['Survived']:
    print(f"  {sex}: {grp.mean()*100:.1f}% survived ({grp.sum()}/{len(grp)})")

print(f"\n{'=' * 60}")
print("SURVIVAL BY PCLASS")
print("=" * 60)
for pc, grp in train.groupby('Pclass')['Survived']:
    print(f"  Class {pc}: {grp.mean()*100:.1f}% survived ({grp.sum()}/{len(grp)})")

print(f"\n{'=' * 60}")
print("SURVIVAL BY EMBARKED")
print("=" * 60)
for emb, grp in train.groupby('Embarked')['Survived']:
    print(f"  {emb}: {grp.mean()*100:.1f}% survived ({grp.sum()}/{len(grp)})")

print(f"\n{'=' * 60}")
print("SURVIVAL BY TITLE (top 6)")
print("=" * 60)
for title in ['Mr', 'Mrs', 'Miss', 'Master', 'Dr', 'Rev']:
    mask = train['Title'] == title
    if mask.sum() > 0:
        rate = train.loc[mask, 'Survived'].mean()
        print(f"  {title}: {rate*100:.1f}% survived (n={mask.sum()})")

print(f"\n{'=' * 60}")
print("STATISTICAL TESTS: FEATURE-TARGET ASSOCIATION")
print("=" * 60)
# Chi-squared for categorical features
for col in ['Sex', 'Pclass', 'Embarked']:
    ct = pd.crosstab(train[col], train['Survived'])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    cramers_v = np.sqrt(chi2 / (len(train) * (min(ct.shape) - 1)))
    print(f"  {col:12s}: chi2={chi2:.2f}, p={p:.2e}, Cramér's V={cramers_v:.3f}")

# Mann-Whitney U for numeric features
for col in ['Age', 'Fare', 'SibSp', 'Parch']:
    g0 = train.loc[train['Survived']==0, col].dropna()
    g1 = train.loc[train['Survived']==1, col].dropna()
    u_stat, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    n0, n1 = len(g0), len(g1)
    r = 1 - (2 * u_stat) / (n0 * n1)
    print(f"  {col:12s}: U={u_stat:.0f}, p={p:.2e}, rank-biserial r={r:.3f}")

## 6. Correlation Analysis

Compute Pearson correlations between numeric features (including encoded Sex and derived features). Identify multicollinearity risks that could affect logistic regression.

In [ ]:
# Derive features for correlation analysis
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
train['HasCabin'] = train['Cabin'].notna().astype(int)
train['Sex_encoded'] = (train['Sex'] == 'female').astype(int)

corr_cols = ['Survived', 'Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch',
             'Fare', 'FamilySize', 'IsAlone', 'HasCabin']
corr = train[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            linewidths=0.5)
plt.title("Correlation Matrix (Numeric + Encoded Features)")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

# Print correlations with target, sorted by absolute value
print("Correlations with Survived (sorted by |r|):")
print("-" * 50)
surv_corr = corr['Survived'].drop('Survived').sort_values(key=abs, ascending=False)
for feat, r in surv_corr.items():
    strength = 'Strong' if abs(r) > 0.4 else ('Moderate' if abs(r) > 0.2 else 'Weak')
    direction = 'Positive' if r > 0 else 'Negative'
    print(f"  {feat:15s}: r={r:+.3f}  ({strength}, {direction})")

print(f"\nMulticollinearity concerns (|r| > 0.5):")
print("-" * 50)
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr.columns[i]:15s} — {corr.columns[j]:15s}: r={r:.3f}")

## 7. Interaction Effects

The Sex x Pclass interaction is expected to be the most informative pattern (from historical accounts: "women and children first" varied by class). Also examine Age bins and Family Size effects.

In [ ]:
# Sex x Pclass interaction — the most critical pattern
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar chart
ct = train.pivot_table('Survived', 'Pclass', 'Sex', aggfunc='mean')
ct.plot(kind='bar', ax=axes[0], color=['#d9534f', '#5cb85c'])
axes[0].set_title('Survival Rate by Sex × Pclass')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Sex')

# Heatmap view
ct2 = train.pivot_table('Survived', 'Sex', 'Pclass', aggfunc='mean')
sns.heatmap(ct2, annot=True, fmt=".1%", cmap="RdYlGn", ax=axes[1], vmin=0, vmax=1,
            linewidths=1)
axes[1].set_title('Survival Rate Heatmap: Sex × Pclass')

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/sex_pclass_interaction.png", dpi=150, bbox_inches='tight')
plt.show()

print("Sex × Pclass survival rates:")
print("-" * 55)
cross = train.groupby(['Sex', 'Pclass'])['Survived'].agg(['mean', 'count'])
for (sex, pc), row in cross.iterrows():
    print(f"  {sex:6s} / Class {pc}: {row['mean']*100:.1f}% survived (n={int(row['count'])})")

print("\nThis is the single most informative pattern in the data.")
print("3rd-class females had a coin-flip survival (50%), while 1st/2nd-class females were ~93-97%.")

In [ ]:
# Age group analysis
age_bins = [0, 5, 12, 18, 35, 60, 80]
age_labels = ['0-5', '6-12', '13-18', '19-35', '36-60', '61-80']
train['AgeBin'] = pd.cut(train['Age'], bins=age_bins, labels=age_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survival by age group
age_surv = train.groupby('AgeBin', observed=True)['Survived'].agg(['mean', 'count'])
age_surv['mean'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Survival Rate by Age Group')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)
for i, (idx, row) in enumerate(age_surv.iterrows()):
    axes[0].text(i, row['mean'] + 0.02, f"n={int(row['count'])}", ha='center', fontsize=9)

# Family size analysis
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
fs_surv = train.groupby('FamilySize')['Survived'].agg(['mean', 'count'])
fs_surv['mean'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Survival Rate by Family Size')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xlabel('Family Size (SibSp + Parch + 1)')
axes[1].set_ylim(0, 1)
for i, (idx, row) in enumerate(fs_surv.iterrows()):
    axes[1].text(i, row['mean'] + 0.02, f"n={int(row['count'])}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/survival_by_family_size.png", dpi=150, bbox_inches='tight')
plt.show()

print("Survival by Age Group:")
for ab, row in age_surv.iterrows():
    print(f"  {ab}: {row['mean']*100:.1f}% survived (n={int(row['count'])})")

age_null = train['Age'].isna()
print(f"\nAge missing → survival rate: {train.loc[age_null, 'Survived'].mean()*100:.1f}% (n={age_null.sum()})")
print(f"Age present → survival rate: {train.loc[~age_null, 'Survived'].mean()*100:.1f}% (n={(~age_null).sum()})")
print("→ Missingness is NOT random — an AgeMissing indicator feature may carry signal")

print(f"\nSurvival by Family Size:")
for fs, row in fs_surv.iterrows():
    print(f"  Size {fs}: {row['mean']*100:.1f}% survived (n={int(row['count'])})")

train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
print(f"\nAlone: {train.loc[train['IsAlone']==1, 'Survived'].mean()*100:.1f}% survived (n={train['IsAlone'].sum()})")
print(f"With family: {train.loc[train['IsAlone']==0, 'Survived'].mean()*100:.1f}% survived (n={(train['IsAlone']==0).sum()})")
print("→ Inverted-U pattern: optimal at size 2-4. Suggests 3-level bin: Small(1), Medium(2-4), Large(5+)")

## 8. Temporal Patterns

**Not applicable.** This dataset captures a single historical event (the sinking of the RMS Titanic on April 15, 1912). There is no temporal dimension — no trends, seasonality, or structural breaks. All passengers experienced the same event simultaneously.

## 9. Subgroup Analysis

Investigate Cabin/Deck, ticket groups, zero-fare passengers, and train vs test distribution alignment.

In [ ]:
# Cabin / Deck analysis
train['HasCabin'] = train['Cabin'].notna().astype(int)
train['Deck'] = train['Cabin'].str[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HasCabin
hc_surv = train.groupby('HasCabin')['Survived'].mean()
hc_surv.plot(kind='bar', ax=axes[0], color=['#d9534f', '#5cb85c'])
axes[0].set_title('Survival Rate: Has Cabin vs No Cabin')
axes[0].set_xticklabels(['No Cabin', 'Has Cabin'], rotation=0)
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)

# By deck
deck_surv = train.groupby('Deck')['Survived'].agg(['mean', 'count']).sort_values('mean', ascending=False)
deck_surv['mean'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Survival Rate by Deck (where known)')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)
for i, (idx, row) in enumerate(deck_surv.iterrows()):
    axes[1].text(i, row['mean'] + 0.02, f"n={int(row['count'])}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Has cabin → survival: {train.loc[train['HasCabin']==1, 'Survived'].mean()*100:.1f}% (n={train['HasCabin'].sum()})")
print(f"No cabin  → survival: {train.loc[train['HasCabin']==0, 'Survived'].mean()*100:.1f}% (n={(train['HasCabin']==0).sum()})")
print("\nSurvival by Deck:")
for deck, row in deck_surv.iterrows():
    print(f"  Deck {deck}: {row['mean']*100:.1f}% survived (n={int(row['count'])})")
print("\n→ HasCabin is largely redundant with Pclass (r=-0.726). Deck adds modest granularity.")

In [ ]:
# Ticket group analysis
ticket_counts = train['Ticket'].value_counts()
train['TicketSize'] = train['Ticket'].map(ticket_counts)

shared_tickets = ticket_counts[ticket_counts > 1]
print(f"Unique tickets: {len(ticket_counts)}")
print(f"Shared tickets (>1 passenger): {len(shared_tickets)}")
print(f"Passengers with shared tickets: {shared_tickets.sum()}")

print(f"\nSurvival by ticket group size (n >= 5):")
for ts, grp in train.groupby('TicketSize')['Survived']:
    if len(grp) >= 5:
        print(f"  Size {ts}: {grp.mean()*100:.1f}% survived (n={len(grp)})")

# Zero-fare passengers
zero_fare = train[train['Fare'] == 0]
print(f"\nZero-fare passengers: {len(zero_fare)}")
print(f"  Survival rate: {zero_fare['Survived'].mean()*100:.1f}%")
print(f"  All male: {(zero_fare['Sex'] == 'male').all()}")
print(f"  Class distribution: {zero_fare['Pclass'].value_counts().to_dict()}")
print("  → Possibly crew, journalists, or special arrangements")

# Train vs Test distribution comparison
print(f"\n{'=' * 60}")
print("TRAIN vs TEST DISTRIBUTION COMPARISON")
print("=" * 60)
for col in ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']:
    if train[col].dtype in ['float64', 'int64'] and col not in ['Pclass']:
        t_mean, te_mean = train[col].mean(), test[col].mean()
        diff_pct = abs(t_mean - te_mean) / t_mean * 100 if t_mean != 0 else 0
        print(f"  {col:12s}: train={t_mean:.2f}, test={te_mean:.2f}, diff={diff_pct:.1f}%")
    else:
        t_dist = train[col].value_counts(normalize=True)
        te_dist = test[col].value_counts(normalize=True)
        max_diff = max(abs(t_dist.get(v, 0) - te_dist.get(v, 0)) for v in t_dist.index)
        print(f"  {col:12s}: max category proportion diff = {max_diff:.3f}")
print("\n→ No significant covariate shift between train and test sets.")

## 10. Key Findings & Feature Hypotheses

### Confirmed Hypotheses
| # | Hypothesis (from 1.3) | Evidence | Implication |
|---|---|---|---|
| 1 | Sex is strongest predictor | Cramer's V=0.541; females 74.2% vs males 18.9% | Must include; "women and children first" confirmed |
| 2 | Class-based lifeboat access | 1st: 63%, 2nd: 47%, 3rd: 24%; Sex x Pclass spans 13.5%-96.8% | Pclass essential; interaction term critical |
| 3 | Missing Cabin is not random | HasCabin correlates with Pclass (r=-0.726) and survival (r=+0.317) | HasCabin useful proxy; Deck adds modest value |
| 4 | Feature interactions important | Sex x Pclass is most informative pattern | Tree models capture natively; logistic regression needs explicit terms |
| 5 | Small dataset favors regularized models | 891 rows; extreme kurtosis in SibSp/Parch | Overfitting risk real; CV essential |

### Surprising Findings
| # | Finding | Evidence | Implication |
|---|---|---|---|
| 1 | Age NOT significant linearly | Mann-Whitney p=0.16; r=-0.077 | Non-linear effect; need bins or interaction with Sex |
| 2 | Family size inverted-U pattern | Size 2-4: 55-72%; Size 1: 30%; Size 5+: 0-20% | 3-level bin better than binary IsAlone |
| 3 | Age missingness correlates with survival | Missing: 29.4% vs Present: 40.6% | AgeMissing indicator carries signal |
| 4 | Zero-fare passengers all male, 6.7% survival | 15 passengers across classes 1-3 | Small but distinct subgroup |

### Feature Hypotheses (Priority Order)
| # | Feature | Source | Rationale | Priority |
|---|---|---|---|---|
| 1 | Title | Name | 15.7% (Mr) to 79.2% (Mrs) survival range | High |
| 2 | FamilySizeBin | SibSp, Parch | Captures non-linear inverted-U pattern | High |
| 3 | log(Fare+1) | Fare | Skew=4.78; normalizes distribution | High |
| 4 | Sex x Pclass | Sex, Pclass | Spans 13.5%-96.8% survival | High (for linear models) |
| 5 | HasCabin | Cabin | 66.7% vs 30.0% survival | Medium |
| 6 | AgeMissing | Age | Non-random missingness signal | Medium |
| 7 | AgeBin | Age | Non-linear age effect | Medium |
| 8 | Deck | Cabin | Granularity beyond HasCabin | Low |
| 9 | TicketGroupSize | Ticket | Mirrors family size pattern | Low |

### Data Leakage Risks
**None identified.** All features represent pre-event passenger attributes. Train/test split uses disjoint PassengerId ranges.

### Modeling Implications
- **Tree ensembles** (RF, GBM) ideal: handle interactions natively, robust to outliers, no scaling needed
- **Logistic Regression** needs explicit Sex*Pclass interaction and feature scaling
- Log-transform Fare, bin Age and FamilySize
- Stratified k-fold CV required (38.4% positive class)